# పాఠం 17 - Foundry Local మరియు Qwen‌తో స్థానిక AI ఏజెంట్లను సృష్టించడం

ఈ నోట్బుక్‌లో మీరు పూర్తిగా మీ వర్క్‌స్టేషన్‌లో నడిచే **స్థానిక ఇంజినీరింగ్ అసిస్టెంట్**ను నిర్మిస్తారు. ఎక్కడా క్లౌడ్ ఇన్‌ఫరెన్స్ ఉపయోగించబడదు. అసిస్టెంట్ చేయగలదు:

1. **Foundry Local ద్వారా Qwen ఫంక్షన్ కాలింగ్** ద్వారా టూల్స్‌ను కాల్ చేయండి.
2. శాండ్బాక్స్ ప్రాజెక్ట్ డైరెక్టరీలోని ఫైళ్లను **జాబితా చేసి చదవండి**.
3. సాదా మెట్రిక్స్‌తో **కోడ్‌ను విశ్లేషించండి**.
4. స్థానిక RAG (Chroma) తో **డాక్యుమెంటేషన్‌ను శోధించండి**.
5. **స్థానిక MCP సర్వర్ ఉపయోగించండి** (ఏదైనా కాన్ఫిగర్ చేయబడలేదంటే మృదువుగా వదిలేయబడుతుంది).

ఏజెంట్ కోడ్ క్లౌడ్ పాఠాలతో చాలా దగ్గరగా ఉంటుంది — కేవలం క్లయింట్ ఎండ్పాయింట్ క్లౌడ్ నుండి `localhost` కి మారుతుంది.


## సెటప్

ఈ నోట్బుక్‌ను నడిపించే ముందు:

1. **Microsoft Foundry Local ను ఇన్‌స్టాల్ చేయండి** (మీ OS కొరకు [డాక్యుమెంటేషన్](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) చూడండి).
2. **Qwen మోడల్‌ను డౌన్‌లోడ్ చేసి స్టార్ట్ చేయండి:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. క్రింద ఇచ్చిన Python ప్యాకేజీలు ఇన్‌స్టాల్ చేయండి.

అన్నీ లోకల్‌గా నడుస్తాయి. సుమారు 8 GB RAM ఉన్న యంత్రం కనిష్టంగా అవసరం.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. ఫౌండ్రీ లోకల్‌కు కనెక్ట్ అవ్వండి

`FoundryLocalManager` అవసరమైతే మోడల్‌ను డౌన్‌లోడ్ చేస్తుంది, లోకల్ సేవను ప్రారంభిస్తుంది, మరియు మాకు ఒక **OpenAI- అనుగుణమైన ఎండ్పాయింట్** అందిస్తుంది. ఆపై మనం సాధారణ OpenAI SDK ని దానిపై పాయింట్ చేస్తాము. API కీ ఒక లోకల్ ప్లేస్‌హోల్డర్ మాత్రమే — మేఘ గుర్తింపులేమీ ఉండవు.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. లోకల్ టూల్స్ (సాండ్‌బాక్స్ చేయబడిన ఫైల్ ఆపరేషన్స్)

మేము డిస్క్‌పై ఒక చిన్న నమూనా ప్రాజెక్టును సృష్టించి, ఆ ప్రాజెక్టు రూట్‌కు సంబంధించిన టూల్స్‌ను నిర్వచిస్తాము. సాండ్‌బాక్స్ తనిఖీ స్థానికంగా కూడా ప్రాధాన్యం కలిగి ఉంది: ఎటువంటి మార్గాలను చదివే టూల్ మీ యూజర్ అనుమతులతో పనిచేస్తుంది మరియు మీరు టచ్ చేయగలిగే ఏదైనా పదును కలిగివుంచవచ్చు.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. స్థానిక RAG తో Chroma

మేము చిన్న డాక్యుమెంటేషన్ స్నిపెట్లను స్థానిక Chroma సేకరణలో ఎम्बెడ్ చేయబడతాము. Chroma ప్రాసెస్‌లో పరుస్తుంది మరియు వెక్టర్లను డిస్క్‌లో నిల్వ చేస్తుంది — సర్వర్ లేదు, క్లౌడ్ లేదు. `search_docs` టూల్ క్వెరీకి సంబందించిన అత్యంత సంబంధిత స్నిపెట్‌లను రిట్రీవ్ చేస్తుంది.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. పరికరం-కాల్ లూప్  

ఇప్పుడ 우리는 OpenAI tools schema ఉపయోగించి పరికరాలను మోడల్‌తో నమోదు చేసి ప్రామాణిక పరికరం-కాల్ లూప్ నడుపుతాము — మోడల్ ఒక పరికరం కోరుతాడు, మనం దాన్ని స్థానికంగా అమలు చేస్తాము, ఫలితాన్ని తిరిగి అందిస్తాము, మరియు మోడల్ తుది సమాధానం ఇచ్చేవరకు పునరావృతం చేస్తాము. Qwen యొక్క నమ్మకమైన ఫంక్షన్ కాలింగ్ ఈ ప్రక్రియను పరికరంపై సాఫీగా చేస్తుంది.  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. లోకల్ MCP (ఐచ్ఛికం)

MCP అనేది క్లౌడ్ సేవ కాకుండా ఒక ట్రాన్స్‌పోర్ట్ — MCP సర్వర్‌ను `stdio` పై లోకల్ ప్రాసెస్‌గా అమలు చేయవచ్చు. మీరు ఒకటి ఏర్పరచుకున్నట్లయితే (ఉదాహరణకు ఫైల్‌సిస్టమ్ సర్వర్) లోకల్ MCP సర్వర్‌కు ఎలా కనెక్ట్ అవ్వాలో క్రింది సెల్ చూపిస్తుంది. `LOCAL_MCP_COMMAND` సెట్ చేయని సందర్భంలో ఇది సురక్షితంగా దాటివిడుస్తుంది, అందువల్ల నోట్‌బుక్ అది లేకుండానే పూర్తిగా నడుస్తుంది.

భద్రతా గమనిక: లోకల్ MCP సర్వర్ మీ యూజర్ అనుమతులతో నడుస్తుంది. దాన్ని ఒక ప్రాజెక్ట్ డైరెక్టరీతో పరిమితం చేసి, దాని అవుట్పుట్లు తీసుకోకమునుపు ధ్రువీకరించండి.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## సారాంశం

మీరు మీ మెషిన్‌లో పూర్తిగా నడిచే ఇంజనీరింగ్ సహాయకుడు ఒకదాన్ని నిర్మించారు:

- **Foundry Local** ఓపెన్AI అనుకూల ఎండ్పాయింట్ వెనుక **Qwen** మోడల్‌ని అందించింది — కాబట్టి ఏజెంట్ కోడ్ క్లౌడ్ పాఠాలతో సరిపోతుంది.
- **Sandboxed tools** ప్రాజెక్ట్ డైరెక్టరీని వదులకుండా ఏజెంట్ ఫైల్ యాక్సెస్ మరియు కోడ్ విశ్లేషణను ఇచ్చాయి.
- **Chroma** డాక్యుమెంటేషన్‌పై **స్థానిక RAG** అందించింది.
- **Local MCP** ఆఫ్‌లైన్‌లో MCP పర్యావరణాన్ని మళ్ళీ ఎలా ఉపయోగించుకోవచ్చో చూపించింది.

ఎప్పుడూ క్లౌడ్ ఇన్ఫరెన్స్ ఉపయోగించలేదు.

### సవాలు

స్థానిక ఏజెంట్‌ను విస్తరించండి:

1. **బహుళ MCP సర్వర్‌లతో పని చేయండి** — ఫైల్సిస్టమ్ సర్వర్ మరియు గిట్ సర్వర్‌ను కనెక్ట్ చేసి ఏజెంట్ వాటిలో ఎంచుకునేలా చేయండి.
2. **స్థానిక మెమొరీ ఉపయోగించండి** — సహాయకుడు నాటుబుక్లను మళ్లీ ప్రారంభించినప్పుడు ఇంతవరకు జరిగిన చదివే సంభాషణ చరిత్రను డిస్క్‌లో నిల్వ చేయండి.
3. **స్థానిక బహుళ-ఏజెంట్ సమన్వయాన్ని మద్దతు ఇవ్వండి** — రెండవ స్థానిక ఏజెంట్(ఉదాహరణకు రివ్యూ చేయేవారు) ను జతచేసి ఇద్దరూ కలిసి ఒక పనిపై పనిచేయండి.

తదుపరి పాఠంలో మీరు ప్రదర్శించిన AI ఏజెంట్‌లను సురక్షితం చేయడం నేర్చుకుంటారు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
